In [10]:
import pandas as pd
from pathlib import Path

# 1. Setup paths (adjust to your project root)
PROJECT_ROOT = Path(r"C:\Users\Vladi\OneDrive\Documents\STGNN_multi-sku")
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "ml_ready_data.parquet"

# 2. Load the data
df = pd.read_parquet(DATA_PATH)

# 3. Recreate the exact index mapping used in the training pipeline
# We must sort by unique_id to match the tensor preparation logic
nodes_metadata = df[['unique_id', 'item_id', 'dept_id', 'store_id']].drop_duplicates().sort_values("unique_id")

# 4. Create a lookup function
def lookup_node(node_index):
    row = nodes_metadata.iloc[node_index]
    return {
        "unique_id": row['unique_id'],
        "item_id": row['item_id'],
        "dept_id": row['dept_id'],
        "store_id": row['store_id']
    }

# Example: Look up Node 0 and Node 144
print(f"Node 0: {lookup_node(0)}")
print(f"Node 584: {lookup_node(584)}")

Node 0: {'unique_id': 'FOODS_1_004_CA_1_evaluation', 'item_id': np.int64(0), 'dept_id': np.int64(0), 'store_id': np.int64(0)}
Node 584: {'unique_id': 'FOODS_3_574_TX_3_evaluation', 'item_id': np.int64(140), 'dept_id': np.int64(2), 'store_id': np.int64(6)}


In [5]:
import numpy as np
import torch
from statsmodels.tsa.stattools import grangercausalitytests

# ==========================================
# 1. GENERATE SYNTHETIC RETAIL DATA
# ==========================================
np.random.seed(42)
torch.manual_seed(42)

T = 500  # 500 days of history

# X represents our "Predictor" item (e.g., Tide Detergent on sale)
X = np.random.normal(0, 1, T)

# Y represents our "Target" item (e.g., Store Brand Detergent)
Y = np.zeros(T)

# Create the physical rule: Y depends on its own past AND X's past
# X heavily Granger-causes Y
for t in range(1, T):
    Y[t] = 0.5 * Y[t-1] + 0.8 * X[t-1] + np.random.normal(0, 0.1)


# ==========================================
# 2. STATSMODELS (CPU BASELINE)
# ==========================================
print("--- STATSMODELS ---")
# statsmodels expects a 2D array where Column 1 is Target (Y), Column 2 is Predictor (X)
data = np.column_stack([Y, X]) 

# Run the test for lag = 1
sm_res = grangercausalitytests(data, maxlag=[1], verbose=False)
sm_f_stat = sm_res[1][0]['ssr_ftest'][0]
print(f"F-Statistic: {sm_f_stat:.6f}")


# ==========================================
# 3. PYTORCH LSTSQ (GPU PROTOTYPE)
# ==========================================
print("\n--- PYTORCH ---")
Y_t = torch.tensor(Y, dtype=torch.float32)
X_t = torch.tensor(X, dtype=torch.float32)

# Create the lagged slices for T=1
Y_target = Y_t[1:].unsqueeze(1)  # [T-1, 1]
Y_lagged = Y_t[:-1].unsqueeze(1) # [T-1, 1]
X_lagged = X_t[:-1].unsqueeze(1) # [T-1, 1]

# CRITICAL: We must add an intercept (column of 1s) or the math will drift!
ones = torch.ones_like(Y_lagged) 

# Build the Design Matrices
A_R = torch.cat([Y_lagged, ones], dim=1)                 # Restricted: Y ~ Y_lag + intercept
A_UR = torch.cat([Y_lagged, X_lagged, ones], dim=1)      # Unrestricted: Y ~ Y_lag + X_lag + intercept

# Solve the Least Squares equation (extracting only the solution coefficients)
beta_R = torch.linalg.lstsq(A_R, Y_target).solution
beta_UR = torch.linalg.lstsq(A_UR, Y_target).solution

# THE FIX: Manually calculate the Sum of Squared Residuals (SSR)
# SSR = sum((Actual - Predicted)^2)
SSR_R = torch.sum((Y_target - (A_R @ beta_R))**2).item()
SSR_UR = torch.sum((Y_target - (A_UR @ beta_UR))**2).item()

# Calculate the F-Statistic
n_obs = Y_target.shape[0]          # Number of observations (T - 1)
num_params_UR = A_UR.shape[1]      # Number of parameters in Unrestricted model (3)
df_denom = n_obs - num_params_UR   # Degrees of freedom for the denominator
df_num = 1                         # Number of restrictions (we dropped 1 feature: X_lag)

f_stat_torch = ((SSR_R - SSR_UR) / df_num) / (SSR_UR / df_denom)
print(f"F-Statistic: {f_stat_torch:.6f}")

# Assert equality
assert np.isclose(sm_f_stat, f_stat_torch, atol=1e-4), "Mismatch detected!"
print("\n✅ VALIDATION PASSED: PyTorch math perfectly matches statsmodels.")

import scipy.stats as stats

# ... [Your existing PyTorch code calculating f_stat_torch] ...

# ==========================================
# 4. THE P-VALUE (Hybrid Transfer)
# ==========================================
# Move the F-stat back to CPU (if it was on GPU) and use SciPy's 
# Survival Function (sf), which is identical to (1 - CDF).
p_value_torch = stats.f.sf(f_stat_torch, df_num, df_denom)
sm_p_value = sm_res[1][0]['ssr_ftest'][1] # Extract the statsmodels p-value

print(f"PyTorch P-Value:     {p_value_torch:.10e}")
print(f"Statsmodels P-Value: {sm_p_value:.10e}")

# The threshold for your Adjacency Matrix
is_significant = p_value_torch < 0.05
print(f"Draw Edge in Graph?: {is_significant}")

--- STATSMODELS ---
F-Statistic: 31425.079067

--- PYTORCH ---
F-Statistic: 31425.077098

✅ VALIDATION PASSED: PyTorch math perfectly matches statsmodels.
PyTorch P-Value:     0.0000000000e+00
Statsmodels P-Value: 0.0000000000e+00
Draw Edge in Graph?: True


In [6]:
import numpy as np
import torch
import scipy.stats as stats
from statsmodels.tsa.stattools import grangercausalitytests

def validate_granger(Y, X, scenario_name):
    print(f"\n{'='*50}")
    print(f"🧪 SCENARIO: {scenario_name}")
    print(f"{'='*50}")
    
    # --- 1. STATSMODELS ---
    data = np.column_stack([Y, X]) 
    sm_res = grangercausalitytests(data, maxlag=[1], verbose=False)
    sm_f_stat = sm_res[1][0]['ssr_ftest'][0]
    sm_p_value = sm_res[1][0]['ssr_ftest'][1]

    # --- 2. PYTORCH ---
    Y_t = torch.tensor(Y, dtype=torch.float32)
    X_t = torch.tensor(X, dtype=torch.float32)

    Y_target = Y_t[1:].unsqueeze(1)
    Y_lagged = Y_t[:-1].unsqueeze(1)
    X_lagged = X_t[:-1].unsqueeze(1)
    ones = torch.ones_like(Y_lagged) 

    A_R = torch.cat([Y_lagged, ones], dim=1)                 
    A_UR = torch.cat([Y_lagged, X_lagged, ones], dim=1)      

    beta_R = torch.linalg.lstsq(A_R, Y_target).solution
    beta_UR = torch.linalg.lstsq(A_UR, Y_target).solution

    SSR_R = torch.sum((Y_target - (A_R @ beta_R))**2).item()
    SSR_UR = torch.sum((Y_target - (A_UR @ beta_UR))**2).item()

    df_denom = Y_target.shape[0] - A_UR.shape[1]   
    df_num = 1                         

    f_stat_torch = ((SSR_R - SSR_UR) / df_num) / (SSR_UR / df_denom)
    p_value_torch = stats.f.sf(f_stat_torch, df_num, df_denom)

    # --- 3. COMPARISON ---
    print(f"Statsmodels P-Value: {sm_p_value:.6f}")
    print(f"PyTorch P-Value:     {p_value_torch:.6f}")
    
    is_match = np.isclose(sm_f_stat, f_stat_torch, atol=1e-4)
    print(f"Match?:              {'✅ YES' if is_match else '❌ NO'}")
    print(f"Draw Edge (p<0.05)?: {'YES (Causal)' if p_value_torch < 0.05 else 'NO (Independent)'}")


# ==========================================
# TEST SCENARIOS
# ==========================================
np.random.seed(100)
T = 500

# 1. STRICT INDEPENDENCE (Should be NO edge)
X_ind = np.random.normal(0, 1, T)
Y_ind = np.zeros(T)
for t in range(1, T):
    Y_ind[t] = 0.5 * Y_ind[t-1] + np.random.normal(0, 0.1) # X does not affect Y
validate_granger(Y_ind, X_ind, "Strict Independence")


# 2. CANNIBALIZATION / NEGATIVE CAUSALITY (Should be YES edge)
X_can = np.random.normal(0, 1, T)
Y_can = np.zeros(T)
for t in range(1, T):
    Y_can[t] = 0.5 * Y_can[t-1] - 0.8 * X_can[t-1] + np.random.normal(0, 0.1) # X drives Y down
validate_granger(Y_can, X_can, "Cannibalization (Negative Causality)")


# 3. REVERSE CAUSALITY (Should be NO edge)
# We make Y cause X. But the test asks "Does X cause Y?"
Y_rev = np.random.normal(0, 1, T)
X_rev = np.zeros(T)
for t in range(1, T):
    X_rev[t] = 0.5 * X_rev[t-1] + 0.8 * Y_rev[t-1] + np.random.normal(0, 0.1) # Y causes X
validate_granger(Y_rev, X_rev, "Reverse Causality (Checking Directionality)")


🧪 SCENARIO: Strict Independence
Statsmodels P-Value: 0.299654
PyTorch P-Value:     0.299656
Match?:              ✅ YES
Draw Edge (p<0.05)?: NO (Independent)

🧪 SCENARIO: Cannibalization (Negative Causality)
Statsmodels P-Value: 0.000000
PyTorch P-Value:     0.000000
Match?:              ✅ YES
Draw Edge (p<0.05)?: YES (Causal)

🧪 SCENARIO: Reverse Causality (Checking Directionality)
Statsmodels P-Value: 0.577194
PyTorch P-Value:     0.577213
Match?:              ✅ YES
Draw Edge (p<0.05)?: NO (Independent)


In [7]:
import numpy as np
import torch
import scipy.stats as stats
from statsmodels.tsa.stattools import grangercausalitytests

# ==========================================
# 1. GENERATE BATCHED SYNTHETIC DATA
# ==========================================
np.random.seed(100)
torch.manual_seed(100)
T = 500

# Scenario 0: Independence
X_0 = np.random.normal(0, 1, T)
Y_0 = np.zeros(T)
for t in range(1, T): Y_0[t] = 0.5 * Y_0[t-1] + np.random.normal(0, 0.1)

# Scenario 1: Cannibalization
X_1 = np.random.normal(0, 1, T)
Y_1 = np.zeros(T)
for t in range(1, T): Y_1[t] = 0.5 * Y_1[t-1] - 0.8 * X_1[t-1] + np.random.normal(0, 0.1)

# Scenario 2: Reverse Causality (Y causes X)
Y_2 = np.random.normal(0, 1, T)
X_2 = np.zeros(T)
for t in range(1, T): X_2[t] = 0.5 * X_2[t-1] + 0.8 * Y_2[t-1] + np.random.normal(0, 0.1)

# Stack into Batched Tensors: Shape [Batch=3, Time=500]
Y_batch = torch.tensor(np.vstack([Y_0, Y_1, Y_2]), dtype=torch.float32)
X_batch = torch.tensor(np.vstack([X_0, X_1, X_2]), dtype=torch.float32)


# ==========================================
# 2. BATCHED PYTORCH CAUSAL DISCOVERY
# ==========================================
print("🚀 RUNNING BATCHED PYTORCH (3 pairs simultaneously)...")

# Slice and add the feature dimension -> Shape: [Batch=3, Time=499, Features=1]
Y_target = Y_batch[:, 1:].unsqueeze(-1)  
Y_lagged = Y_batch[:, :-1].unsqueeze(-1) 
X_lagged = X_batch[:, :-1].unsqueeze(-1) 
ones = torch.ones_like(Y_lagged)

# Build Batched Design Matrices
A_R = torch.cat([Y_lagged, ones], dim=-1)                # Shape: [3, 499, 2]
A_UR = torch.cat([Y_lagged, X_lagged, ones], dim=-1)     # Shape: [3, 499, 3]

# Solve 3 regressions simultaneously
beta_R = torch.linalg.lstsq(A_R, Y_target).solution      # Shape: [3, 2, 1]
beta_UR = torch.linalg.lstsq(A_UR, Y_target).solution    # Shape: [3, 3, 1]

# Calculate Residuals using Batched Matrix Multiplication (bmm)
# A_UR (3x499x3) @ beta_UR (3x3x1) = Predictions (3x499x1)
pred_R = torch.bmm(A_R, beta_R)
pred_UR = torch.bmm(A_UR, beta_UR)

# Sum of Squared Residuals per batch element (sum across Time and Features)
SSR_R = torch.sum((Y_target - pred_R)**2, dim=[1, 2])    # Shape: [3]
SSR_UR = torch.sum((Y_target - pred_UR)**2, dim=[1, 2])  # Shape: [3]

# Degrees of freedom
df_num = 1
df_denom = Y_target.shape[1] - A_UR.shape[2] # 499 - 3 = 496

# Calculate F-stats and P-values arrays
f_stat_batch = ((SSR_R - SSR_UR) / df_num) / (SSR_UR / df_denom)
p_values_batch = stats.f.sf(f_stat_batch.numpy(), df_num, df_denom)


# ==========================================
# 3. STATSMODELS SEQUENTIAL BASELINE
# ==========================================
print("🐌 RUNNING STATSMODELS (Sequentially)...\n")
scenarios = ["0: Independence", "1: Cannibalization", "2: Reverse Causality"]

for i in range(3):
    print(f"{'='*50}\n🧪 SCENARIO {scenarios[i]}")
    
    # Run Statsmodels
    data = np.column_stack([Y_batch[i].numpy(), X_batch[i].numpy()]) 
    sm_res = grangercausalitytests(data, maxlag=[1], verbose=False)
    sm_p_value = sm_res[1][0]['ssr_ftest'][1]
    
    # Extract corresponding PyTorch result
    pt_p_value = p_values_batch[i]
    
    print(f"Statsmodels P-Value: {sm_p_value:.6f}")
    print(f"PyTorch P-Value:     {pt_p_value:.6f}")
    print(f"Match?:              {'✅ YES' if np.isclose(sm_p_value, pt_p_value, atol=1e-4) else '❌ NO'}")

🚀 RUNNING BATCHED PYTORCH (3 pairs simultaneously)...
🐌 RUNNING STATSMODELS (Sequentially)...

🧪 SCENARIO 0: Independence
Statsmodels P-Value: 0.299654
PyTorch P-Value:     0.299656
Match?:              ✅ YES
🧪 SCENARIO 1: Cannibalization
Statsmodels P-Value: 0.000000
PyTorch P-Value:     0.000000
Match?:              ✅ YES
🧪 SCENARIO 2: Reverse Causality
Statsmodels P-Value: 0.577194
PyTorch P-Value:     0.577213
Match?:              ✅ YES
